# 命名實體識別 (NER) — 基於 Transformers 的序列標注

## 學習目標

1. 理解 BIO 標記體系與 token-level 標注任務的本質
2. 掌握「word → subword」對齊問題的正確處理方式（word_ids 映射 + -100 遮罩）
3. 使用 HuggingFace `Trainer` + 完整 `TrainingArguments` 訓練序列標注模型
4. 以 `seqeval` 計算 entity-level precision / recall / F1
5. 理解 2026 載入慣例：`device_map='auto'`、`torch.bfloat16`、`safetensors`

## 前置知識

- 建議先完成 `01-Intro-tasks/` 的 pipeline 與 fine-tuning 基礎章節
- 理解 attention mask 與 subword tokenization 概念

## 相鄰 Notebook

- 上一個：`../01-text_classification/text_classification.ipynb` — 句子級分類
- 下一個：`../03-question_answering/qa.ipynb` — 抽取式問答（span 定位）

---

**資料集**：`peoples_daily_ner`（人民日報 NER，中文，標注 PER / ORG / LOC）  
**基底模型**：`hfl/chinese-macbert-base`（哈工大訊飛中文 BERT 變體）

## 版本鎖定與環境安裝

全 repo 統一版本基線，確保可重現性。

- `transformers>=4.46`：`eval_strategy`、`save_safetensors` 等新參數需要此版本
- `datasets>=3.0`：`map(batched=True, num_proc=N)` 效能改進
- `evaluate>=0.4`：`seqeval` 整合
- `accelerate>=1.0`：`device_map='auto'` 後端
- `seqeval`：entity-level 序列評估指標

In [ ]:
# Install — pin versions for reproducibility
%pip install -q \
    "transformers[torch]>=4.46" \
    "datasets>=3.0" \
    "evaluate>=0.4" \
    "accelerate>=1.0" \
    "seqeval>=1.2" \
    "safetensors>=0.4"

## Step 1 — 載入套件與設定可重現亂數種子

`set_seed(42)` 讓 PyTorch、NumPy、Python random 三者同步，確保每次執行結果相同。

In [ ]:
import numpy as np
import torch
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)
from transformers import pipeline as hf_pipeline

set_seed(42)

print(f"torch  : {torch.__version__}")
print(f"device : {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2 — 載入資料集

`peoples_daily_ner` 是 HuggingFace Hub 上的標準中文 NER 資料集，直接以 `load_dataset` 從 Hub 下載，無須掛載 Google Drive 或使用本機硬路徑。

若環境無法連網，可改用：
```python
from datasets import DatasetDict
ner_datasets = DatasetDict.load_from_disk("/path/to/local/ner_data")
```

In [ ]:
ner_datasets = load_dataset("peoples_daily_ner")
ner_datasets

### 原始樣本結構

每筆樣本有兩個欄位：
- `tokens`：已分詞的 token 串列（中文以字為單位）
- `ner_tags`：對應的標籤 ID 串列（整數）

注意：原始標籤**不含** `[CLS]` / `[SEP]` 等特殊 token，稍後預處理時需對齊。

In [ ]:
# Inspect raw sample — note there are no special tokens yet
ner_datasets["train"][0]

### B-I-O 標記體系

NER 採用 **BIO（Begin-Inside-Outside）** 標記方案：

| 標籤 | 意義 | 範例 |
|------|------|------|
| `O` | 非實體 token | "在"、"工作" |
| `B-PER` | 人名實體**開始** | "李"（李小龍的第一字）|
| `I-PER` | 人名實體**內部** | "小"、"龍" |
| `B-ORG` | 組織名**開始** | "联"（联合国的第一字）|
| `I-ORG` | 組織名**內部** | "合"、"国" |
| `B-LOC` | 地點名**開始** | "长"（长城的第一字）|
| `I-LOC` | 地點名**內部** | "城" |

`seqeval` 在計算指標時會將 `B-X I-X I-X ...` 視為一個完整實體 span，這正是 entity-level F1 的基礎。

In [ ]:
# Inspect dataset features and label schema
print("Features:")
print(ner_datasets["train"].features)

# Build label list from dataset metadata (single source of truth)
label_list = ner_datasets["train"].features["ner_tags"].feature.names
print(f"\nLabel list ({len(label_list)} classes): {label_list}")

# Build id2label / label2id mappings — persist into model config later
id2label = {idx: label for idx, label in enumerate(label_list)}
label2id = {label: idx for idx, label in enumerate(label_list)}
print(f"\nid2label: {id2label}")

## Step 3 — 資料預處理：Word → Subword 標籤對齊

這是 NER 任務的核心難點。Tokenizer 可能把一個 word 切成多個 subword piece（WordPiece/BPE），而原始標籤是 word-level，必須做對齊。

### 以 `word_ids()` 映射標籤

```
word tokens  : [李 小 龍] → word_id = [0, 1, 2]
subword ids  : [CLS 李 小 龍 SEP] → word_id = [None, 0, 1, 2, None]
label ids    : [B-PER I-PER I-PER] → [-100, B-PER, I-PER, I-PER, -100]
```

規則：
- `word_id is None`（特殊 token）→ label = **-100**（CrossEntropyLoss 預設忽略 -100）
- 其餘 word_id → 直接取該 word 的 NER label

### 動態 Padding 說明

`process_function` 中**不做 padding**，只做 truncation。實際 padding 交由 `DataCollatorForTokenClassification` 在 batch 組裝時動態執行，讓每個 batch 以該 batch 最長序列為準，比固定 max_length 節省約 20-40% 計算量。

In [ ]:
# Load tokenizer
MODEL_ID = "hfl/chinese-macbert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Quick demo: how word_ids works for subword alignment
print("=== Subword alignment demo ===")
sample_tokens = ner_datasets["train"][0]["tokens"]
sample_encoding = tokenizer(sample_tokens, is_split_into_words=True)
print(f"Input tokens  : {sample_tokens[:6]}")
print(f"word_ids      : {sample_encoding.word_ids()[:10]}")
print(f"Decoded input : {tokenizer.decode(sample_encoding['input_ids'])}")

In [ ]:
# English subword split demo
eng_enc = tokenizer("interesting word")
print("English subword split:")
print(f"  tokens   : {tokenizer.convert_ids_to_tokens(eng_enc['input_ids'])}")
print(f"  word_ids : {eng_enc.word_ids()}")
# Note: 'interesting' may be split into 'interest' + '##ing'
# Each piece shares the same word_id -> both get the same NER label

In [ ]:
def process_function(examples):
    """
    Tokenize pre-tokenized word sequences and align NER labels to subword tokens.

    Key points:
    - is_split_into_words=True: input is already a list of words, not a raw string
    - truncation=True, no padding: DataCollatorForTokenClassification handles dynamic padding
    - Special tokens (word_id is None) get label -100 so CrossEntropyLoss ignores them
    - Subword pieces inherit the label of their parent word
    """
    tokenized_examples = tokenizer(
        examples["tokens"],
        max_length=128,
        truncation=True,
        is_split_into_words=True,
        # No padding here — DataCollatorForTokenClassification pads per batch (dynamic padding)
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_examples.word_ids(batch_index=i)
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                # Special tokens ([CLS], [SEP], [PAD]) -> -100: ignored by loss
                label_ids.append(-100)
            else:
                # Subword pieces inherit their parent word's label
                label_ids.append(label[word_id])
        labels.append(label_ids)
    tokenized_examples["labels"] = labels
    return tokenized_examples


# Apply with batched=True for 3-5x speedup over row-by-row processing
tokenized_datasets = ner_datasets.map(
    process_function,
    batched=True,
    num_proc=2,  # set to os.cpu_count() on machines with many cores
    remove_columns=ner_datasets["train"].column_names,
)
print(tokenized_datasets)
print("\nSample after tokenization:")
print(tokenized_datasets["train"][0])

### 驗證標籤對齊

確認 -100 的位置（特殊 token）與實際標籤的分佈正確無誤。

In [ ]:
# Sanity check: verify label alignment for sample 0
sample = tokenized_datasets["train"][0]
ids = sample["input_ids"]
lbls = sample["labels"]
tokens = tokenizer.convert_ids_to_tokens(ids)
print(f"{'token':<12} {'label_id':>10} {'label_str':>12}")
print("-" * 40)
for tok, lbl in zip(tokens, lbls):
    lbl_str = id2label.get(lbl, "[special]") if lbl != -100 else "[ignore]"
    print(f"{tok:<12} {lbl:>10} {lbl_str:>12}")

## Step 4 — 建立模型

### 2026 載入慣例

```python
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
)
```

| 參數 | 理由 |
|------|------|
| `device_map='auto'` | Accelerate 自動分配 GPU/CPU/disk，多卡環境自動切 tensor parallelism |
| `torch_dtype=torch.bfloat16` | bf16 與 fp32 動態範圍相同（指數位數相同），不像 fp16 有數值溢出風險；Ampere+ GPU 原生支援 bf16 矩陣乘法 |
| `use_safetensors=True` | Safetensors 格式不含 Python pickle，無法執行任意程式碼，載入速度快 2-5x，支援 mmap 零拷貝 |
| `id2label` / `label2id` 在載入時傳入 | 讓 `model.config` 直接包含標籤映射，pipeline 和 push_to_hub 不需再手動設定 |

**VRAM 估算（macbert-base，7 類）：**
- bf16：約 180 MB（基底模型 102M 參數 × 2 bytes）
- 訓練時加梯度 + AdamW 優化器狀態：約 500-700 MB
- 建議最低 2 GB VRAM，或使用 CPU 訓練（速度較慢）

In [ ]:
# Load model with 2026 conventions
# - id2label/label2id baked into config at load time
# - bf16 for memory efficiency (safe on Ampere+; falls back to fp32 on older GPU)
# - use_safetensors=True for secure, fast weight loading
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
    # device_map='auto' is handled by Trainer/Accelerate automatically;
    # setting it here can conflict with Trainer's DDP setup on multi-GPU.
    # For single-GPU or CPU, Trainer moves the model automatically.
)
print(f"Model config num_labels : {model.config.num_labels}")
print(f"id2label                : {model.config.id2label}")
print(f"Model dtype             : {next(model.parameters()).dtype}")

## Step 5 — 建立評估函數

`seqeval` 是序列標注任務的標準評估庫，計算 **entity-level**（而非 token-level）指標：
- 只有完整 span（B + 所有 I）完全正確才算 True Positive
- 比 token-level accuracy 更嚴格，更接近實際應用需求

### 評估函數設計要點

1. `predictions` 是 logits（batch × seq_len × num_labels），取 `argmax` 得 label ID
2. 過濾掉 `label == -100` 的位置（特殊 token）
3. 將 ID 轉回字串標籤，再傳給 `seqeval`
4. 回傳 precision、recall、F1 供 `Trainer` 追蹤最佳模型

In [ ]:
# Load seqeval from evaluate hub
seqeval_metric = evaluate.load("seqeval")


def compute_metrics(pred):
    """
    Compute entity-level NER metrics using seqeval.

    seqeval treats 'B-X I-X I-X' as a single entity span;
    a span is correct only if the entire span matches (strict mode).
    This is harder than token-level accuracy and reflects real-world utility.
    """
    logits, labels = pred  # logits: (batch, seq_len, num_labels)
    predictions = np.argmax(logits, axis=-1)  # (batch, seq_len)

    # Remove special tokens (label == -100) and convert IDs to string labels
    true_predictions = [
        [label_list[p] for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    result = seqeval_metric.compute(
        predictions=true_predictions,
        references=true_labels,
        mode="strict",  # entire span must match
        scheme="IOB2",
    )

    # Return per-entity-type and overall metrics
    return {
        "precision": result["overall_precision"],
        "recall": result["overall_recall"],
        "f1": result["overall_f1"],
        "accuracy": result["overall_accuracy"],
    }

## Step 6 — TrainingArguments：2026 完整慣例

### 重要參數說明

| 參數 | 值 | 理由 |
|------|-----|------|
| `bf16=True` | True | 相比 fp16 不易數值溢出；Trainer 自動偵測 GPU 支援 |
| `warmup_ratio=0.1` | 0.1 | 前 10% steps 線性 warmup，避免初期大梯度破壞預訓練權重 |
| `lr_scheduler_type='cosine'` | cosine | Cosine decay 使學習率平滑下降，比 linear 在後期收斂更穩定 |
| `optim='adamw_torch_fused'` | fused | PyTorch 2.0+ 的 fused AdamW 核心，比逐步更新快 10-20% |
| `max_grad_norm=1.0` | 1.0 | 梯度裁剪，防止梯度爆炸（NLP 任務標準值）|
| `eval_strategy='steps'` | steps | 每 N steps 評估一次，比每 epoch 更早發現過擬合 |
| `save_safetensors=True` | True | 儲存為 safetensors 格式（非 pickle）|
| `seed=42` | 42 | 與 `set_seed(42)` 一致，確保 DataLoader shuffle 可重現 |

### Effective Batch Size

**effective_batch = per_device_train_batch_size × gradient_accumulation_steps × num_gpus**

本範例：64 × 1 × 1 = 64。若 VRAM 不足，可改 `per_device_train_batch_size=16, gradient_accumulation_steps=4`，效果等價。

In [ ]:
# 2026 TrainingArguments — all key parameters set explicitly
args = TrainingArguments(
    output_dir="models_for_ner",
    # Batch size and accumulation
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    gradient_accumulation_steps=1,  # effective batch = 64 * 1 = 64
    # Training schedule
    num_train_epochs=3,
    learning_rate=2e-5,
    warmup_ratio=0.1,              # linear warmup for first 10% of steps
    lr_scheduler_type="cosine",    # cosine decay after warmup
    max_grad_norm=1.0,             # gradient clipping
    # Precision and optimizer
    bf16=True,                     # bfloat16: same dynamic range as fp32, no overflow risk
    optim="adamw_torch_fused",     # PyTorch 2.0+ fused kernel, ~15% faster than default
    # Evaluation and checkpointing
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    metric_for_best_model="f1",
    greater_is_better=True,
    load_best_model_at_end=True,
    # Logging
    logging_steps=100,
    logging_dir="./logs",
    # Reproducibility and serialization
    seed=42,
    save_safetensors=True,         # save in safetensors format (not pickle)
    report_to="none",              # disable wandb/tensorboard for this demo
)

## Step 7 — 建立 Trainer

`DataCollatorForTokenClassification` 負責：
1. 將同一 batch 內長度不同的序列動態 pad 到該 batch 最長長度
2. 同步 pad `labels`，pad 位置填 -100（loss 忽略）
3. 生成 `attention_mask`

這比在 `process_function` 裡固定 pad 到 128 節省約 30% GPU 計算量。

In [ ]:
# DataCollatorForTokenClassification: dynamic padding per batch
# - pads input_ids, attention_mask, token_type_ids to batch max length
# - pads labels with -100 so padded positions are ignored by loss
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,  # pad length to multiple of 8 for tensor core efficiency
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    tokenizer=tokenizer,  # needed for push_to_hub and saving tokenizer config
)

## Step 8 — 模型訓練與評估

訓練過程中 Trainer 會：
- 每 100 steps 記錄 loss
- 每 500 steps 在 validation set 評估，輸出 precision / recall / F1
- 儲存 F1 最高的 checkpoint
- 訓練結束後自動載入最佳 checkpoint（`load_best_model_at_end=True`）

In [ ]:
# Train
# VRAM requirement: ~700 MB for macbert-base in bf16 + optimizer states
# If OOM: reduce per_device_train_batch_size or switch to CPU (slow)
train_result = trainer.train()

# Log final training metrics
print("\n=== Training complete ===")
print(f"train/loss             : {train_result.training_loss:.4f}")
print(f"total steps            : {train_result.global_step}")

In [ ]:
# Evaluate on held-out test set
test_metrics = trainer.evaluate(eval_dataset=tokenized_datasets["test"])
print("\n=== Test set evaluation ===")
for k, v in test_metrics.items():
    print(f"  {k:<30} {v:.4f}" if isinstance(v, float) else f"  {k:<30} {v}")

## Step 9 — 模型推論

### 2026 Pipeline 載入慣例

```python
ner_pipe = pipeline(
    "token-classification",
    model=model,          # model already has id2label in config from Step 4
    tokenizer=tokenizer,
    device_map="auto",    # works on CPU / single-GPU / multi-GPU automatically
    aggregation_strategy="simple",
)
```

`device_map='auto'` 由 Accelerate 後端自動選擇可用裝置，無須傳入裝置索引。`aggregation_strategy="simple"` 將相鄰的 `B-X I-X I-X` 合併為一個實體 span，回傳 `entity_group`、`score`、`start`、`end`。

In [ ]:
# model.config already has id2label baked in from Step 4
# No need to set it again here
print("id2label in config:", model.config.id2label)

# Build pipeline with device_map='auto'
# aggregation_strategy='simple': merge B-X I-X ... spans into single entities
ner_pipe = hf_pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",          # auto-selects GPU if available, else CPU
    aggregation_strategy="simple",
)

In [ ]:
# Inference demo
test_text = "小明在北京上班，同時在清華大學攻讀博士學位。"
results = ner_pipe(test_text)

print(f"Input: {test_text}\n")
print(f"{'Entity':<10} {'Group':<8} {'Score':>8}  Span")
print("-" * 45)
for r in results:
    span = test_text[r["start"]:r["end"]]
    print(f"{span:<10} {r['entity_group']:<8} {r['score']:>8.4f}  [{r['start']}:{r['end']}]")

In [ ]:
# Group results by entity type for a structured output
def extract_entities(text: str, pipe) -> dict[str, list[str]]:
    """
    Run NER pipeline and group detected entities by type.

    Args:
        text: input string
        pipe: HuggingFace token-classification pipeline with aggregation_strategy set

    Returns:
        dict mapping entity_group -> list of surface strings
    """
    raw = pipe(text)
    result: dict[str, list[str]] = {}
    for item in raw:
        group = item["entity_group"]
        surface = text[item["start"]:item["end"]]
        result.setdefault(group, []).append(surface)
    return result


entities = extract_entities(test_text, ner_pipe)
for etype, spans in entities.items():
    print(f"  {etype}: {spans}")

## Step 10 — 儲存模型與 Model Card（可選）

訓練完成後，將最佳 checkpoint 以 safetensors 格式儲存，並設定最小 model card metadata，方便之後用 `push_to_hub` 分享。

`safe_serialization=True` 確保儲存格式為 safetensors（非 pickle），與 `save_safetensors=True` 保持一致。

In [ ]:
import os
from pathlib import Path

# Save best model (loaded by load_best_model_at_end=True)
output_path = Path("models_for_ner/best")
output_path.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(output_path))  # saves with safe_serialization=True per TrainingArguments
tokenizer.save_pretrained(str(output_path))

print(f"Model saved to: {output_path.resolve()}")
print("Files:", [f.name for f in output_path.iterdir()])

In [ ]:
# Optional: push to HuggingFace Hub
# Uncomment and set HF_TOKEN in environment to enable
#
# import os
# hf_repo_id = "your-username/chinese-macbert-base-ner-peoples-daily"
# trainer.push_to_hub(
#     repo_id=hf_repo_id,
#     commit_message="Add peoples_daily_ner fine-tuned model",
#     language="zh",
#     license="apache-2.0",
#     tags=["token-classification", "ner", "chinese", "macbert"],
# )
# print(f"Model pushed to: https://huggingface.co/{hf_repo_id}")
print("push_to_hub skipped — set HF_TOKEN and uncomment to publish.")

## 小結

### 本 Notebook 涵蓋的核心概念

1. **BIO 標記體系**：B（Begin）、I（Inside）、O（Outside）定義實體邊界，`seqeval` 在 entity span 層級計算 F1。

2. **Word → Subword 對齊**：用 `word_ids()` 映射標籤是 token classification 的核心技巧；特殊 token 填 -100 讓 loss 自動忽略。

3. **動態 Padding**：`DataCollatorForTokenClassification` 在 batch 層級 pad，比固定 max_length 節省大量 GPU 計算。

4. **2026 載入慣例**：`device_map='auto'` 讓 Accelerate 自動分配裝置；`torch.bfloat16` 提供與 fp32 相同的動態範圍且記憶體用量減半；`use_safetensors=True` 確保無 pickle 安全載入。

5. **完整 TrainingArguments**：`bf16`、`warmup_ratio`、`cosine` scheduler、`adamw_torch_fused`、`eval_strategy`、`save_safetensors`。

6. **Pipeline 慣例**：`device_map='auto'` 自動選擇可用裝置；`id2label` 在模型載入時設定，pipeline 直接繼承，無須額外設定。

---

### 練習題

1. **更換模型**：將 `hfl/chinese-macbert-base` 換成 `hfl/chinese-roberta-wwm-ext`，比較 test F1 差異。

2. **首個 I token 策略**：目前所有 subword piece 都繼承父 word 的標籤。試著將每個 word 的**第一個** subword 保留原標籤，**後續 piece** 改為 -100（只對首 piece 計算 loss），重新訓練並觀察結果。

3. **詳細指標**：修改 `compute_metrics` 以回傳每個實體類型（PER / ORG / LOC）的個別 F1，找出哪一類最難分。

4. **長序列**：`peoples_daily_ner` 最長句子超過 128 tokens 嗎？改用 `max_length=256`，重新訓練並對比結果。

5. **錯誤分析**：在 test set 上找出預測錯誤的樣本（特別是 B-X 預測正確但 I-X 預測錯誤的 span），分析錯誤模式。

---

**下一步**：`../03-question_answering/qa.ipynb` — 抽取式問答也是 token-level 任務（start/end span 預測），與 NER 的 word_ids 技巧有相通之處。